In [4]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor

DATA_DIR = Path("data")
OUT_DIR = Path("outputs_winner")
OUT_DIR.mkdir(exist_ok=True)

def read_csv_safe(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_csv(path, engine="python")

def parse_bool_series(s):
    if pd.api.types.is_bool_dtype(s):
        return s.astype(int)
    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .map({"true": 1, "false": 0, "1": 1, "0": 0, "yes": 1, "no": 0})
        .fillna(0)
        .astype(int)
    )

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def build_dataset():
    match_df = read_csv_safe(DATA_DIR / "gold_match.csv")
    context_df = read_csv_safe(DATA_DIR / "gold_match_context.csv")
    tickets_df = read_csv_safe(DATA_DIR / "gold_match_tickets.csv")

    match_df["match_date"] = pd.to_datetime(match_df["match_date"], errors="coerce")
    match_df["is_home_match"] = parse_bool_series(match_df["is_home_match"])
    match_df["tickets_scanned"] = pd.to_numeric(match_df["tickets_scanned"], errors="coerce")

    if context_df is not None:
        for c in ["has_promotion", "is_weekend", "is_midweek", "is_public_holiday", "is_school_holiday_flanders"]:
            if c in context_df.columns:
                context_df[c] = parse_bool_series(context_df[c])

    if tickets_df is not None:
        for c in tickets_df.columns:
            if c != "match_id":
                tickets_df[c] = pd.to_numeric(tickets_df[c], errors="coerce")

    home_df = match_df[match_df["is_home_match"] == 1].copy()

    if context_df is not None:
        keep_context_cols = [c for c in context_df.columns if c != "match_date"]
        home_df = home_df.merge(context_df[keep_context_cols], on="match_id", how="left")

    if tickets_df is not None:
        home_df = home_df.merge(tickets_df, on="match_id", how="left")

    home_df = home_df[home_df["tickets_scanned"].notna()].copy()
    home_df["opponent"] = home_df["away_team"]
    home_df = home_df.sort_values("match_date").reset_index(drop=True)

    home_df["prev_home_attendance"] = home_df["tickets_scanned"].shift(1)
    home_df["home_attendance_avg_3"] = home_df["tickets_scanned"].shift(1).rolling(3, min_periods=1).mean()
    home_df["home_attendance_avg_5"] = home_df["tickets_scanned"].shift(1).rolling(5, min_periods=1).mean()

    opp_avg = []
    for i in range(len(home_df)):
        hist = home_df.iloc[:i]
        opp = home_df.loc[i, "opponent"]
        same_opp = hist[hist["opponent"] == opp]["tickets_scanned"]
        opp_avg.append(same_opp.mean() if len(same_opp) > 0 else np.nan)

    home_df["opp_attendance_avg"] = opp_avg

    fill_value = home_df["tickets_scanned"].mean()
    home_df["prev_home_attendance_filled"] = home_df["prev_home_attendance"].fillna(fill_value)
    home_df["home_attendance_avg_3_filled"] = home_df["home_attendance_avg_3"].fillna(fill_value)
    home_df["home_attendance_avg_5_filled"] = home_df["home_attendance_avg_5"].fillna(fill_value)
    home_df["opp_attendance_avg_filled"] = home_df["opp_attendance_avg"].fillna(fill_value)

    home_df["baseline_blend"] = (
        0.5 * home_df["prev_home_attendance_filled"] +
        0.3 * home_df["home_attendance_avg_3_filled"] +
        0.2 * home_df["opp_attendance_avg_filled"]
    )

    home_df["month"] = home_df["match_date"].dt.month
    home_df["year"] = home_df["match_date"].dt.year

    return home_df

def evaluate_baseline(df):
    splitter = TimeSeriesSplit(n_splits=5)
    rows = []
    preds = []

    for fold, (_, test_idx) in enumerate(splitter.split(df), start=1):
        y_test = df.iloc[test_idx]["tickets_scanned"].astype(float)
        y_pred = df.iloc[test_idx]["baseline_blend"].astype(float)

        rows.append(
            {
                "model": "baseline_blend",
                "fold": fold,
                "mae": mean_absolute_error(y_test, y_pred),
                "rmse": rmse(y_test, y_pred),
                "r2": r2_score(y_test, y_pred),
            }
        )

        part = df.iloc[test_idx][["match_date", "match_id", "opponent", "tickets_scanned"]].copy()
        part["model"] = "baseline_blend"
        part["prediction"] = y_pred.values
        preds.append(part)

    fold_df = pd.DataFrame(rows)
    summary = pd.DataFrame(
        [
            {
                "model": "baseline_blend",
                "mae_mean": fold_df["mae"].mean(),
                "mae_std": fold_df["mae"].std(),
                "rmse_mean": fold_df["rmse"].mean(),
                "rmse_std": fold_df["rmse"].std(),
                "r2_mean": fold_df["r2"].mean(),
                "r2_std": fold_df["r2"].std(),
            }
        ]
    )

    return summary, pd.concat(preds, ignore_index=True)

def evaluate_models(df, feature_cols, model_defs):
    data = df[["match_date", "match_id", "opponent", "tickets_scanned"] + feature_cols].copy()
    data = data.sort_values("match_date").reset_index(drop=True)

    X = data[feature_cols].copy()
    y = data["tickets_scanned"].astype(float).copy()

    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    num_cols = [c for c in X.columns if c not in cat_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ]
    )

    splitter = TimeSeriesSplit(n_splits=5)
    summary_rows = []
    pred_frames = []

    for model_name, model in model_defs.items():
        fold_rows = []
        fold_parts = []

        for fold, (train_idx, test_idx) in enumerate(splitter.split(X), start=1):
            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]
            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]

            pipe = Pipeline([("prep", preprocessor), ("model", model)])
            pipe.fit(X_train, y_train)
            y_pred = pipe.predict(X_test)

            fold_rows.append(
                {
                    "model": model_name,
                    "fold": fold,
                    "mae": mean_absolute_error(y_test, y_pred),
                    "rmse": rmse(y_test, y_pred),
                    "r2": r2_score(y_test, y_pred),
                }
            )

            part = data.iloc[test_idx][["match_date", "match_id", "opponent", "tickets_scanned"]].copy()
            part["model"] = model_name
            part["prediction"] = y_pred
            fold_parts.append(part)

        fold_df = pd.DataFrame(fold_rows)
        summary_rows.append(
            {
                "model": model_name,
                "mae_mean": fold_df["mae"].mean(),
                "mae_std": fold_df["mae"].std(),
                "rmse_mean": fold_df["rmse"].mean(),
                "rmse_std": fold_df["rmse"].std(),
                "r2_mean": fold_df["r2"].mean(),
                "r2_std": fold_df["r2"].std(),
            }
        )
        pred_frames.append(pd.concat(fold_parts, ignore_index=True))

    return pd.DataFrame(summary_rows).sort_values("mae_mean").reset_index(drop=True), pd.concat(pred_frames, ignore_index=True)

def make_metrics_bar_chart(metrics_df):
    plot_df = metrics_df.copy()

    plt.figure(figsize=(10, 6))
    plt.bar(plot_df["model"], plot_df["mae_mean"])
    plt.title("Model comparison by MAE")
    plt.xlabel("Model")
    plt.ylabel("MAE")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "metrics_bar_chart_mae.png", dpi=150)
    plt.close()

    plt.figure(figsize=(10, 6))
    plt.bar(plot_df["model"], plot_df["rmse_mean"])
    plt.title("Model comparison by RMSE")
    plt.xlabel("Model")
    plt.ylabel("RMSE")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "metrics_bar_chart_rmse.png", dpi=150)
    plt.close()

def make_best_model_plot(preds_df, best_model):
    best_df = preds_df[preds_df["model"] == best_model].copy().sort_values("match_date")

    plt.figure(figsize=(12, 6))
    plt.plot(best_df["match_date"], best_df["tickets_scanned"], marker="o", label="Actual attendance")
    plt.plot(best_df["match_date"], best_df["prediction"], marker="o", label=f"Predicted attendance ({best_model})")
    plt.title(f"Actual vs Predicted Attendance - {best_model}")
    plt.xlabel("Match date")
    plt.ylabel("Tickets scanned")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "actual_vs_predicted_best_model.png", dpi=150)
    plt.close()

def make_all_models_plot(preds_df):
    pivot_actual = preds_df[["match_date", "tickets_scanned"]].drop_duplicates().sort_values("match_date")

    plt.figure(figsize=(13, 7))
    plt.plot(pivot_actual["match_date"], pivot_actual["tickets_scanned"], marker="o", linewidth=2, label="Actual")

    for model_name in preds_df["model"].unique():
        model_df = preds_df[preds_df["model"] == model_name].copy().sort_values("match_date")
        plt.plot(model_df["match_date"], model_df["prediction"], marker="o", linestyle="--", label=model_name)

    plt.title("Actual vs Predicted Attendance - All Models")
    plt.xlabel("Match date")
    plt.ylabel("Tickets scanned")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "actual_vs_predicted_all_models.png", dpi=150)
    plt.close()

def make_residual_boxplot(preds_df):
    plot_df = preds_df.copy()
    plot_df["absolute_error"] = (plot_df["tickets_scanned"] - plot_df["prediction"]).abs()

    models = list(plot_df["model"].unique())
    data = [plot_df.loc[plot_df["model"] == model, "absolute_error"].values for model in models]

    plt.figure(figsize=(11, 6))
    plt.boxplot(data, tick_labels=models)
    plt.title("Absolute Error Distribution by Model")
    plt.xlabel("Model")
    plt.ylabel("Absolute error")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "residual_boxplot.png", dpi=150)
    plt.close()

def make_colorful_summary_chart(metrics_df):
    plot_df = metrics_df.copy().sort_values("mae_mean").reset_index(drop=True)

    colors = ["#2E86DE", "#E74C3C", "#27AE60", "#9B59B6", "#F39C12", "#16A085", "#D35400", "#7F8C8D"]

    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.bar(plot_df["model"], plot_df["mae_mean"], color=colors[:len(plot_df)], edgecolor="black", linewidth=1.2)

    ax.set_title("Attendance Prediction Model Comparison", fontsize=16, fontweight="bold")
    ax.set_xlabel("Model", fontsize=12)
    ax.set_ylabel("MAE", fontsize=12)
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", linestyle="--", alpha=0.35)

    for i, (_, row) in enumerate(plot_df.iterrows()):
        ax.text(i, row["mae_mean"] + 15, f'{row["mae_mean"]:.1f}', ha="center", va="bottom", fontsize=10, fontweight="bold")

    best_idx = plot_df.index[plot_df["model"] == plot_df.iloc[0]["model"]][0]
    bars[best_idx].set_linewidth(2.5)
    bars[best_idx].set_edgecolor("#111111")

    baseline_row = plot_df[plot_df["model"] == "baseline_blend"]
    if not baseline_row.empty:
        baseline_value = baseline_row["mae_mean"].iloc[0]
        ax.axhline(baseline_value, color="#C0392B", linestyle=":", linewidth=2, label=f'Baseline = {baseline_value:.1f}')

    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "colorful_model_comparison.png", dpi=150)
    plt.close()

def main():
    df = build_dataset()

    feature_cols = [
        "baseline_blend",
        "prev_home_attendance_filled",
        "home_attendance_avg_3_filled",
        "opp_attendance_avg_filled",
        "tickets_sold_total",
        "tickets_sold_b2c",
        "tickets_sold_b2b",
        "tickets_trib1",
        "tickets_trib3",
        "tickets_trib4",
    ]

    model_defs = {
        "extra_trees_best": ExtraTreesRegressor(
            n_estimators=800,
            max_depth=5,
            min_samples_leaf=2,
            random_state=42,
        ),
        "gradient_boosting_alt": GradientBoostingRegressor(
            n_estimators=200,
            learning_rate=0.03,
            max_depth=2,
            loss="absolute_error",
            random_state=42,
        ),
        "random_forest_alt": RandomForestRegressor(
            n_estimators=600,
            max_depth=4,
            min_samples_leaf=3,
            random_state=42,
        ),
    }

    baseline_metrics, baseline_preds = evaluate_baseline(df)
    model_metrics, model_preds = evaluate_models(df, feature_cols, model_defs)

    all_metrics = pd.concat([baseline_metrics, model_metrics], ignore_index=True).sort_values("mae_mean").reset_index(drop=True)
    all_preds = pd.concat([baseline_preds, model_preds], ignore_index=True)

    all_metrics.to_csv(OUT_DIR / "comparison_metrics.csv", index=False)
    all_preds.to_csv(OUT_DIR / "predictions.csv", index=False)
    df.to_csv(OUT_DIR / "model_input_dataset.csv", index=False)

    best_model = all_metrics.iloc[0]["model"]
    best_mae = float(all_metrics.iloc[0]["mae_mean"])
    baseline_mae = float(all_metrics.loc[all_metrics["model"] == "baseline_blend", "mae_mean"].iloc[0])

    make_metrics_bar_chart(all_metrics)
    make_best_model_plot(all_preds, best_model)
    make_all_models_plot(all_preds)
    make_residual_boxplot(all_preds)

    print("\nWINNER SEARCH")
    print(all_metrics.to_string(index=False))

    print("\nBEST MODEL:", best_model)
    print("BEST MAE:", round(best_mae, 3))
    print("BASELINE MAE:", round(baseline_mae, 3))
    print("IMPROVEMENT VS BASELINE:", round(baseline_mae - best_mae, 3))
    print("\nSaved graphs:")
    print(OUT_DIR / "metrics_bar_chart_mae.png")
    print(OUT_DIR / "metrics_bar_chart_rmse.png")
    print(OUT_DIR / "actual_vs_predicted_best_model.png")
    print(OUT_DIR / "actual_vs_predicted_all_models.png")
    print(OUT_DIR / "residual_boxplot.png")

    make_colorful_summary_chart(all_metrics)



if __name__ == "__main__":
    main()


WINNER SEARCH
                model    mae_mean    mae_std   rmse_mean   rmse_std   r2_mean   r2_std
     extra_trees_best 1183.664767 315.775217 1415.606083 366.667906 -0.215081 0.751653
gradient_boosting_alt 1189.258649 297.488968 1376.208255 421.138944 -0.057808 0.489727
       baseline_blend 1295.685676 407.522737 1550.153123 502.419634 -0.253964 0.428601
    random_forest_alt 1377.827309 420.812081 1591.754164 517.424779 -0.569145 1.223450

BEST MODEL: extra_trees_best
BEST MAE: 1183.665
BASELINE MAE: 1295.686
IMPROVEMENT VS BASELINE: 112.021

Saved graphs:
outputs_winner/metrics_bar_chart_mae.png
outputs_winner/metrics_bar_chart_rmse.png
outputs_winner/actual_vs_predicted_best_model.png
outputs_winner/actual_vs_predicted_all_models.png
outputs_winner/residual_boxplot.png
